# EDA

## 1. Setup

In [1]:
import pandas as pd
from sqlalchemy import create_engine, types
from sqlalchemy import text # to be able to pass string

### .env file

In [2]:
# Let's load values from the .env file
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

## 2. Reading in prep_tables

In [3]:
# Now building the URL with the values from the .env file

url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

# without specifying the schema default connection is to the schema `public`
# url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

In [4]:
engine = create_engine(url, echo=False)

In [5]:
my_schema = 'crush' # update it to your schema

with engine.begin() as conn: 
    result = conn.execute(text(f'SET search_path TO {my_schema};'))

In [6]:
df_flights = pd.read_sql("SELECT * FROM prep_flights;", con=engine)
df_weather = pd.read_sql("SELECT * FROM prep_weather;", con=engine)


### Dataframes for prep_tables

In [9]:
df_flights

,flight_date,origin,dest,dep_time,arr_time,sched_dep_time,sched_arr_time,dep_delay,arr_delay,cancelled,diverted,distance_miles,distance_km,air_time_minutes
0,2003-02-01,LAS,PHL,36.0,823.0,44,815,-8.0,8.0,0,0,2176,3501.933627,252.0
1,2003-02-01,PHL,ATL,519.0,722.0,520,732,-1.0,-10.0,0,0,665,1070.214091,100.0
2,2003-02-01,BGR,LGA,535.0,713.0,530,710,5.0,3.0,0,0,378,608.332220,75.0
3,2003-02-01,BOS,PHL,532.0,655.0,530,654,2.0,1.0,0,0,280,450.616459,65.0
4,2003-02-01,MHT,PHL,527.0,648.0,530,658,-3.0,-10.0,0,0,290,466.709904,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63413,2003-02-28,PHX,PHL,2241.0,440.0,2245,510,-4.0,-30.0,0,0,2075,3339.389833,220.0
63414,2003-02-28,CVG,LGA,2249.0,34.0,2250,38,-1.0,-4.0,0,0,585,941.466531,75.0
63415,2003-02-28,LAS,IAD,2319.0,624.0,2315,635,4.0,-11.0,0,0,2066,3324.905733,220.0
63416,2003-02-28,OAK,IAD,2320.0,711.0,2320,715,0.0,-4.0,0,0,2409,3876.910895,270.0


In [10]:
df_weather

,airport_code,station_id,timestamp,temp_c,dewpoint_c,humidity_perc,precipitation_mm,snow_mm,wind_direction,wind_speed_kmh,...,date,time,hour,month_name,weekday,date_day,date_month,date_year,cw,day_part
0,BWI,72406,2003-02-07 00:00:00,0.0,-6.6,61.0,0.0,None,130.0,9.4,...,2003-02-07,00:00:00,00:00,february,friday,7.0,2.0,2003.0,6.0,night
1,BWI,72406,2003-02-07 01:00:00,-1.7,-2.8,92.0,NaN,None,170.0,7.6,...,2003-02-07,01:00:00,01:00,february,friday,7.0,2.0,2003.0,6.0,night
2,BWI,72406,2003-02-07 02:00:00,-1.7,-1.7,100.0,1.0,None,0.0,0.0,...,2003-02-07,02:00:00,02:00,february,friday,7.0,2.0,2003.0,6.0,night
3,BWI,72406,2003-02-07 03:00:00,-1.7,-1.7,100.0,1.5,None,0.0,0.0,...,2003-02-07,03:00:00,03:00,february,friday,7.0,2.0,2003.0,6.0,night
4,BWI,72406,2003-02-07 04:00:00,-1.7,-1.7,100.0,1.5,None,110.0,5.4,...,2003-02-07,04:00:00,04:00,february,friday,7.0,2.0,2003.0,6.0,night
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1915,LGA,72503,2003-02-26 19:00:00,-7.2,-14.9,54.0,0.0,None,50.0,18.7,...,2003-02-26,19:00:00,19:00,february,wednesday,26.0,2.0,2003.0,9.0,evening
1916,LGA,72503,2003-02-26 20:00:00,-6.1,-13.9,54.0,0.0,None,70.0,14.8,...,2003-02-26,20:00:00,20:00,february,wednesday,26.0,2.0,2003.0,9.0,evening
1917,LGA,72503,2003-02-26 21:00:00,-6.7,-12.8,62.0,0.0,None,50.0,14.8,...,2003-02-26,21:00:00,21:00,february,wednesday,26.0,2.0,2003.0,9.0,evening
1918,LGA,72503,2003-02-26 22:00:00,-6.1,-12.2,62.0,0.0,None,50.0,13.0,...,2003-02-26,22:00:00,22:00,february,wednesday,26.0,2.0,2003.0,9.0,evening


## 3. Analysis Alex

In [8]:
print("Total flights in March 2003:", len(df_flights))
print("Cancelled:", df_flights['cancelled'].sum())
print("Diverted:", df_flights['diverted'].sum())



Total flights in March 2003: 63418
Cancelled: 6670
Diverted: 276


In [ ]:
merged = pd.merge(
    df_flights, 
    df_weather, 
    left_on='flight_date', right_on='date', 
    how='inner'
)

merged[['dep_delay', 'temp_c', 'precipitation_mm']].corr()

,dep_delay,temp_c,precipitation_mm
dep_delay,1.000000,-0.008730,0.042744
temp_c,-0.008730,1.000000,0.044027
precipitation_mm,0.042744,0.044027,1.000000


## 4. Analysis Janina